In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelGP/ModelGP_RBF.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 1.9815179213957204, 'n_it': 1.1472171016286523}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 300

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.QuasirandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[14.861995708381844, 14.814166927593375, 14.370377701764438, 15.063812939121284, 14.942113838064536, 14.509719297226908, 15.112312366387108, 14.444058764022824, 14.376059258454294, 14.601438538549278, 14.911849201889734, 15.082798426786972, 15.07334947272644, 15.115344538412717, 15.032473774289825, 14.86478057629383, 14.785391181570715, 14.200506923020031, 15.079264951308591, 14.371958960962301, 15.036285633701475, 14.939894214299821, 14.907427701794834, 14.888585798353517, 14.936434802485747, 15.000661996067336, 14.87196304408855, 15.024106985217008, 15.055763672072189, 14.987481313892946, 15.136613830010635, 15.094775403056516, 14.46597538685719, 14.783821821006795, 15.006610192497497, 14.972658725593963, 15.109357375478428, 14.332124775383694, 14.777523530274951, 14.654831271849208, 15.128941453175326, 14.839898804624818, 14.851105284648215, 14.890005189041101, 14.698953973257455, 14.852431220856461, 14.828564850260392, 14.96155299923609, 15.009866192170996, 15.001203428582855, 15.0

In [5]:
np.average(y_max_arr)

np.float64(14.827177102354984)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_RBF/DataGenerated/quasirandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)